In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

FILE_NAME = "p0.8_mu0.2_1"
emb_path = f"pretrain/{FILE_NAME}_node_feat.emb"
interaction_path = f"syn_data/{FILE_NAME}.csv" 
K = 5

with open(emb_path, "r", encoding="utf-8") as f:
    first_line = f.readline().strip().split()

has_header = False
if len(first_line) == 2:
    try:
        int(first_line[0])
        int(first_line[1])
        has_header = True
    except ValueError:
        has_header = False

if has_header:
    emb_df = pd.read_csv(emb_path, sep=r"\s+", skiprows=1, header=None)
else:
    emb_df = pd.read_csv(emb_path, sep=r"\s+", header=None)

node_ids = emb_df.iloc[:, 0].values
X = emb_df.iloc[:, 1:].values.astype(np.float32)

try:
    node_ids = node_ids.astype(int)
except:
    pass


kmeans = KMeans(n_clusters=K, n_init=10, random_state=0)
labels = kmeans.fit_predict(X)

node2label = pd.DataFrame({
    "node": node_ids,
    "label": labels
})

inter_df = pd.read_csv(interaction_path)
inter_df = inter_df[["source", "destination", "timestamp"]].copy()

try:
    inter_df["source"] = inter_df["source"].astype(node2label["node"].dtype)
    inter_df["destination"] = inter_df["destination"].astype(node2label["node"].dtype)
except:
    pass


src_label_df = node2label.rename(columns={
    "node": "source",
    "label": "source_commu"
})
result_df = inter_df.merge(src_label_df, on="source", how="left")


dst_label_df = node2label.rename(columns={
    "node": "destination",
    "label": "destination_commu"
})
result_df = result_df.merge(dst_label_df, on="destination", how="left")


result_df = result_df[
    ["source", "destination", "timestamp", "source_commu", "destination_commu"]
].copy()


out_dir = os.path.join("results", FILE_NAME)
os.makedirs(out_dir, exist_ok=True)

out_path = os.path.join(out_dir, "ctdne.csv")
result_df.to_csv(out_path, index=False)

print(result_df.head(20))
print(f"saved to {out_path}")
print("rows =", len(result_df))
print("missing source_commu =", result_df["source_commu"].isna().sum())
print("missing destination_commu =", result_df["destination_commu"].isna().sum())

    source  destination  timestamp  source_commu  destination_commu
0       16           19          5             4                  4
1        6           46         11             4                  4
2       16           28         18             4                  4
3        5           22         25             1                  4
4        8           35         31             0                  4
5       12           28         37             1                  4
6       15           41         43             4                  1
7        7           49         52             4                  4
8       38           43         60             3                  3
9       24           45         63             2                  4
10       0           45         70             4                  4
11       6           47         77             4                  4
12       6           40         83             4                  2
13      15           21         87             4